# Figure 3: Overview of STARR-FISH Data (Experiment 3)

This notebook analyzes the quality and reproducibility of STARR-FISH data from Experiment 3.

## Overview
- **Experiment 3**: Two brain sections analyzed independently
- **Goal**: Assess reproducibility of CRE and T7 measurements across sections
- **Key metrics**: Cell type correlations, CRE correlations, infection rates

## Contents
1. Setup and data loading
2. T7 vs AAV library size correlation
3. Cell type count reproducibility
4. CRE and T7 expression reproducibility
5. Negative control analysis
6. Heatmaps of expression across cell types
7. Section-to-section correlation analysis
8. Infection rate filtering analysis

## 1. Setup and Imports

In [1]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning, module="docrep")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns
import scanpy as sc
from scipy.stats import pearsonr, linregress
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Add current path to sys.path
import sys
import os
try:
    PWD = os.path.dirname(os.path.abspath(__file__))
except NameError:
    PWD = '/gpfs/commons/groups/ren_lab/guojiezhong/starr-fish/Mouse_brain.Guojie'
sys.path.append(f'{PWD}/')
os.chdir(PWD)

from STARRFISH import STARRFISH
import re

Seed set to 0


Last run with scvi-tools version: 1.3.0


## 2. Helper Functions

Utility functions for reloading modules, dropping test results, and preprocessing data.

In [2]:
def reload(starrfish):
    """Reload STARRFISH modules and update class."""
    import importlib
    import STARRFISH
    importlib.reload(STARRFISH)
    from STARRFISH import STARRFISH
    starrfish.__class__ = STARRFISH
    return starrfish

def drop_test(starrfish, test_method):
    """Remove cached test results from STARRFISH object."""
    if hasattr(starrfish, f'{test_method}_configs'):
        delattr(starrfish, f'{test_method}_configs')
    if hasattr(starrfish, f'{test_method}_results'):
        delattr(starrfish, f'{test_method}_results')
    return starrfish

def preprocess(adata_path):
    """Preprocess AnnData object: extract FOV, clean cell type labels, process CRE info."""
    if type(adata_path) is str:
        adata = sc.read_h5ad(adata_path)
    elif type(adata_path) is sc.AnnData:
        adata = adata_path
    
    # Extract FOV from index
    adata.obs['fov'] = adata.obs.index.str.split('--').str[0]
    
    # Clean cell type labels
    adata.obs['subclass'] = adata.obs['subclass_name'].str.replace('^[0-9]+ ', '', regex=True)
    adata.obs['class'] = adata.obs['class_name'].str.replace('^[0-9]+ ', '', regex=True)
    
    # Fix empty best_subclass
    adata.uns['CRE_info']['best_subclass'][adata.uns['CRE_info']['best_subclass'] == ''] = \
        adata.uns['CRE_info']['label'][adata.uns['CRE_info']['best_subclass'] == ''].copy()
    
    # Process enhancer coordinates
    chrom, start, end = [], [], []
    for i in adata.uns['CRE_info']['enh']:
        if i.startswith('chr'):
            chrom.append(i.split(':')[0])
            start.append(int(re.split('−|-', i.split(':')[1])[0]))
            end.append(int(re.split('−|-', i.split(':')[1])[1]))
        else:
            chrom.append(i)
            start.append('')
            end.append('')
    
    adata.uns['CRE_info']['Chrom'] = pd.Series(chrom).astype(str)
    adata.uns['CRE_info']['Start'] = pd.Series(start).astype(str)
    adata.uns['CRE_info']['End'] = pd.Series(end).astype(str)
    adata.uns['CRE_info']['enh'] = adata.uns['CRE_info']['Chrom'] + ':' + \
                                     adata.uns['CRE_info']['Start'] + '-' + \
                                     adata.uns['CRE_info']['End']
    
    # Clean best_subclass labels
    adata.uns['CRE_info']['best_subclass'] = adata.uns['CRE_info']['best_subclass'].str.replace('_', ' ')
    
    # Rename CREs
    adata.uns['CRE_info'].index = ['CRE' + str(i+1).zfill(3) for i in range(len(adata.uns['CRE_info']))]
    adata.obsm['CRE'] = adata.obsm['CRE'][adata.uns['CRE_info'].index]
    if 'T7CRE' in adata.obsm.keys():
        adata.obsm['T7CRE'] = adata.obsm['T7CRE'][adata.uns['CRE_info'].index]
    
    return adata

## 3. Load STARR-FISH Data

Load pre-processed STARR-FISH objects for Experiment 3:
- **Section 1**: First tissue section
- **Section 2**: Second tissue section
- **Combined**: Both sections merged

In [3]:
starrfish3_sec1 = STARRFISH.load('results/starrfish3_sec1.pkl')
starrfish3_sec2 = STARRFISH.load('results/starrfish3_sec2.pkl')
starrfish3 = STARRFISH.load('results/starrfish3.pkl')

## 4. Define Cell Types and CRE Filters

Set up filters for:
- Cell type annotations and naming conventions
- CRE blacklist (problematic barcodes)
- Cell types with sufficient cells and negative control counts

In [4]:
# Load cell type annotation mappings
subclass_annotation = pd.read_excel(f'Data/abc_atlas/allen_institute_nominature.xlsx')
subclass_annotation['subclass'] = subclass_annotation['subclass_id_label'].str.replace('^[0-9]+ ', '', regex=True)
subclass_annotation['subclass'] = subclass_annotation['subclass'].str.replace('/', '-', regex=True)
subclass_to_subclass_name = subclass_annotation['subclass_id_label'].groupby(subclass_annotation['subclass']).first().to_dict()
subclass_name_to_subclass = subclass_annotation['subclass'].groupby(subclass_annotation['subclass_id_label']).first().to_dict()

# Define CRE blacklist and whitelist
cre_blacklist = ['CRE061', 'CRE143', 'CRE001']
cre_whitelist = starrfish3_sec1.get_creinfo().index[~starrfish3_sec1.get_creinfo().index.isin(cre_blacklist)]

In [5]:
# Define cell types with sufficient coverage
negative_control_cres = starrfish3_sec1.get_negative_control_cres()
cell_types_counts1 = starrfish3_sec1.get_celltypes().value_counts()
cell_types_counts2 = starrfish3_sec2.get_celltypes().value_counts()
cell_types_to_use_1 = cell_types_counts1[cell_types_counts1 > 500].index
cell_types_to_use_2 = cell_types_counts2[cell_types_counts2 > 500].index
cell_types_to_use = cell_types_to_use_1.intersection(cell_types_to_use_2)

# Filter by negative control counts
negative_control_counts1 = starrfish3_sec1.get_cre_expression()[negative_control_cres].groupby(starrfish3_sec1.get_celltypes()).sum()
negative_control_counts2 = starrfish3_sec2.get_cre_expression()[negative_control_cres].groupby(starrfish3_sec2.get_celltypes()).sum()
negative_control_sum_counts1 = starrfish3_sec1.get_cre_expression()[negative_control_cres].sum(axis=1).groupby(starrfish3_sec1.get_celltypes()).sum()
negative_control_sum_counts2 = starrfish3_sec2.get_cre_expression()[negative_control_cres].sum(axis=1).groupby(starrfish3_sec2.get_celltypes()).sum()
common_cell_types_sum_20_nc = negative_control_sum_counts1[negative_control_sum_counts1 > 20].index.intersection(
    negative_control_sum_counts2[negative_control_sum_counts2 > 20].index)

cell_types_to_use_nc_1 = negative_control_sum_counts1[negative_control_sum_counts1 > 10].index
cell_types_to_use_nc_2 = negative_control_sum_counts2[negative_control_sum_counts2 > 10].index
cell_types_to_use_nc = cell_types_to_use_nc_1.intersection(cell_types_to_use_nc_2)
target_cres = starrfish3_sec1.get_creinfo().index[starrfish3_sec1.get_creinfo()['best_subclass'].isin(cell_types_to_use_nc_2)]

print(f"Cell types (>500 cells): {len(cell_types_to_use)}")
print(f"Cell types (NC>10): {len(cell_types_to_use_nc)}")
print(f"Target CREs: {len(target_cres)}")

Cell types (>500 cells): 44
Cell types (NC>10): 52
Target CREs: 160


In [6]:
# Filter by barcode mismatch percentage
mismatching_cres = pd.read_csv('Data/AAV_ONT_Barcode_Counts_vs_Mismatch_Percentage.csv', index_col=0)
cre_whitelist = cre_whitelist[~cre_whitelist.isin(mismatching_cres.index[mismatching_cres['MismatchPercent'] > 20])]
cre_blacklist = np.unique(cre_blacklist + mismatching_cres.index[mismatching_cres['MismatchPercent'] > 20].tolist()).tolist()
print(f"CREs in whitelist: {len(cre_whitelist)}")
print(f"CREs in blacklist: {len(cre_blacklist)}")

CREs in whitelist: 389
CREs in blacklist: 11


## 5. T7 Correlation with AAV Library Size

Analyze the relationship between T7 barcode counts and AAV library size.
Higher library size should correlate with higher T7 detection.

In [7]:
t7_counts = starrfish3.get_t7_expression().sum(axis=0)
t7_counts = t7_counts.loc[starrfish3.lib_size.index]  # align with library size

fig, ax = plt.subplots(ncols=2, figsize=(12, 6))

# Raw scale
sns.scatterplot(x=starrfish3.lib_size['counts'], y=t7_counts, ax=ax[0], alpha=0.5)
ax[0].set_xlabel('AAV library size')
ax[0].set_ylabel('Total T7 counts in all cells')
slope, intercept, r_value, p_value, std_err = linregress(starrfish3.lib_size['counts'], t7_counts)
x = np.linspace(starrfish3.lib_size['counts'].min(), starrfish3.lib_size['counts'].max(), 100)
y = slope * x + intercept
ax[0].plot(x, y, color='red', label=f'Correlation: {r_value:.2f}, p-value: {p_value:.2e}')
ax[0].legend()

# Log scale
t7_counts_log = np.log1p(t7_counts)
sns.scatterplot(x=np.log(starrfish3.lib_size['counts']), y=t7_counts_log, ax=ax[1], alpha=0.5)
ax[1].set_xlabel('Log(AAV library size)')
ax[1].set_ylabel('Log(T7 counts)')
slope_log, intercept_log, r_value_log, p_value_log, std_err_log = linregress(
    np.log(starrfish3.lib_size['counts']), t7_counts_log)
x_log = np.linspace(np.log(starrfish3.lib_size['counts']).min(), 
                     np.log(starrfish3.lib_size['counts']).max(), 100)
y_log = slope_log * x_log + intercept_log
ax[1].plot(x_log, y_log, color='red', label=f'Correlation: {r_value_log:.2f}, p-value: {p_value_log:.2e}')
ax[1].legend()

# Highlight negative controls
sns.scatterplot(x=starrfish3.lib_size['counts'].loc[negative_control_cres], 
                y=t7_counts.loc[negative_control_cres], ax=ax[0], alpha=0.5, color='orange')
sns.scatterplot(x=np.log(starrfish3.lib_size['counts'].loc[negative_control_cres]), 
                y=np.log(t7_counts.loc[negative_control_cres]), ax=ax[1], alpha=0.5, color='orange')

fig.savefig('results/expr3/T7_libsize.pdf')

## 6. Cell Type Count Reproducibility

Compare the number of cells per cell type between Section 1 and Section 2.

In [8]:
cell_type_counts = pd.DataFrame(
    index=cell_types_counts2.index.intersection(cell_types_counts1.index).intersection(starrfish3.get_celltypes()), 
    columns=['Sec1', 'Sec2', 'Exp3'])
cell_type_counts['Exp3'] = starrfish3.get_celltypes().value_counts().loc[cell_type_counts.index]
cell_type_counts['Sec1'] = cell_types_counts1[cell_type_counts.index]
cell_type_counts['Sec2'] = cell_types_counts2[cell_type_counts.index]

fig, ax = plt.subplots(figsize=(10, 4))
sns.scatterplot(x=cell_type_counts['Sec1'], y=cell_type_counts['Sec2'], ax=ax, alpha=0.5)
ax.set_xlabel('Cell type counts in Sec1')
ax.set_ylabel('Cell type counts in Sec2')

# Calculate correlation
to_test = cell_type_counts.index[cell_type_counts['Sec1'] > 0].intersection(
    cell_type_counts.index[cell_type_counts['Sec2'] > 0])
corr, p_value = pearsonr(np.log(cell_type_counts['Sec1'].loc[to_test]), 
                          np.log(cell_type_counts['Sec2'].loc[to_test]))
ax.text(0.05, 0.95, f'Pearson r: {corr:.2f}\np-value: {p_value:.2e}', 
        transform=ax.transAxes, fontsize=12, verticalalignment='top')
ax.set_xscale('log')
ax.set_yscale('log')

fig.savefig('results/expr3/sec1_sec2_cell_type_counts.pdf')

## 7. CRE and T7 Expression Reproducibility

Compare average CRE and T7 counts per cell type between sections.

In [9]:
fig, ax = plt.subplots(ncols=2, figsize=(12, 6))

# Calculate average counts per cell type
cre_counts_sec1 = starrfish3_sec1.get_cre_expression().sum(axis=1)
cre_counts_sec2 = starrfish3_sec2.get_cre_expression().sum(axis=1)
t7_counts_sec1 = starrfish3_sec1.get_t7_expression().sum(axis=1)
t7_counts_sec2 = starrfish3_sec2.get_t7_expression().sum(axis=1)

cre_celltype_sec1 = cre_counts_sec1.groupby(starrfish3_sec1.get_tag('obs:subclass_name')).mean()
cre_celltype_sec2 = cre_counts_sec2.groupby(starrfish3_sec2.get_tag('obs:subclass_name')).mean()
t7_celltype_sec1 = t7_counts_sec1.groupby(starrfish3_sec1.get_tag('obs:subclass_name')).mean()
t7_celltype_sec2 = t7_counts_sec2.groupby(starrfish3_sec2.get_tag('obs:subclass_name')).mean()

common_celltypes = t7_celltype_sec1.index.intersection(t7_celltype_sec2.index)

# T7 counts
sns.scatterplot(x=t7_celltype_sec1.loc[common_celltypes], 
                y=t7_celltype_sec2.loc[common_celltypes], ax=ax[0], alpha=0.5)
to_test = t7_celltype_sec1.index[t7_celltype_sec1 > 0].intersection(
    t7_celltype_sec2.index[t7_celltype_sec2 > 0])
corr, p_value = pearsonr(np.log(t7_celltype_sec1.loc[to_test]), 
                          np.log(t7_celltype_sec2.loc[to_test]))
ax[0].text(0.05, 0.95, f'Pearson r: {corr:.2f}\np-value: {p_value:.2e}', 
           transform=ax[0].transAxes, fontsize=12, verticalalignment='top')
ax[0].set_xscale('log')
ax[0].set_yscale('log')
ax[0].set_xlabel('Average T7 counts per cell type in Sec1')
ax[0].set_ylabel('Average T7 counts per cell type in Sec2')

# CRE counts
sns.scatterplot(x=cre_celltype_sec1.loc[common_celltypes], 
                y=cre_celltype_sec2.loc[common_celltypes], ax=ax[1], alpha=0.5)
to_test = cre_celltype_sec1.index[cre_celltype_sec1 > 0].intersection(
    cre_celltype_sec2.index[cre_celltype_sec2 > 0])
corr, p_value = pearsonr(np.log(cre_celltype_sec1.loc[to_test]), 
                          np.log(cre_celltype_sec2.loc[to_test]))
ax[1].text(0.05, 0.95, f'Pearson r: {corr:.2f}\np-value: {p_value:.2e}', 
           transform=ax[1].transAxes, fontsize=12, verticalalignment='top')
ax[1].set_xlabel('Average CRE counts per cell type in Sec1')
ax[1].set_ylabel('Average CRE counts per cell type in Sec2')
ax[1].set_xscale('log')
ax[1].set_yscale('log')

fig.savefig('results/expr3/sec1_sec2_cre_t7_counts.pdf')

## 8. Negative Control Analysis

Examine negative control CRE and T7 counts across cell types to assess background levels.

In [10]:
fig, ax = plt.subplots(ncols=3, figsize=(15, 4))

for idx, p in enumerate(['CRE', 'T7', 'CRE/T7']):
    if p == 'CRE':
        counts_sec1 = starrfish3_sec1.get_cre_expression()[negative_control_cres].sum(axis=1).groupby(
            starrfish3_sec1.get_tag('obs:subclass')).mean()
        counts_sec2 = starrfish3_sec2.get_cre_expression()[negative_control_cres].sum(axis=1).groupby(
            starrfish3_sec2.get_tag('obs:subclass')).mean()
        ax_work = ax[0]
    elif p == 'T7':
        counts_sec1 = starrfish3_sec1.get_t7_expression()[negative_control_cres].sum(axis=1).groupby(
            starrfish3_sec1.get_tag('obs:subclass')).mean()
        counts_sec2 = starrfish3_sec2.get_t7_expression()[negative_control_cres].sum(axis=1).groupby(
            starrfish3_sec2.get_tag('obs:subclass')).mean()
        ax_work = ax[1]
    elif p == 'CRE/T7':
        counts_sec1 = starrfish3_sec1.get_cre_expression()[negative_control_cres].sum(axis=1).groupby(
            starrfish3_sec1.get_tag('obs:subclass')).mean() / \
            starrfish3_sec1.get_t7_expression()[negative_control_cres].sum(axis=1).groupby(
            starrfish3_sec1.get_tag('obs:subclass')).mean()
        counts_sec2 = starrfish3_sec2.get_cre_expression()[negative_control_cres].sum(axis=1).groupby(
            starrfish3_sec2.get_tag('obs:subclass')).mean() / \
            starrfish3_sec2.get_t7_expression()[negative_control_cres].sum(axis=1).groupby(
            starrfish3_sec2.get_tag('obs:subclass')).mean()
        ax_work = ax[2]
    
    common_celltypes = counts_sec1.index.intersection(counts_sec2.index)
    to_test = counts_sec1.index[(counts_sec1 > 0) & np.isfinite(counts_sec1)].intersection(
        counts_sec2.index[(counts_sec2 > 0) & np.isfinite(counts_sec2)])
    corr, p_value = pearsonr(np.log(counts_sec1.loc[to_test]), 
                              np.log(counts_sec2.loc[to_test]))
    sns.scatterplot(x=counts_sec1.loc[common_celltypes], 
                    y=counts_sec2.loc[common_celltypes], ax=ax_work, alpha=0.5)
    ax_work.text(0.05, 0.95, f'Pearson r: {corr:.2f}\np-value: {p_value:.2e}', 
                 transform=ax_work.transAxes, fontsize=12, verticalalignment='top')
    ax_work.set_xscale('log')
    ax_work.set_yscale('log')
    ax_work.set_xlabel(f'Average NC {p} counts per cell type in Sec1')
    ax_work.set_ylabel(f'Average NC {p} counts per cell type in Sec2')

fig.tight_layout()
fig.savefig('results/expr3/negative_control_comparison.pdf')

## 9. Heatmap Visualization

Create heatmaps showing CRE and T7 expression across cell types.

In [11]:
# Prepare CRE annotations
cell_types_to_use = cell_type_counts.index[(cell_type_counts['Exp3'] > 1000)].map(subclass_to_subclass_name)
cre_anno = pd.DataFrame(data=0, 
                        index=cell_types_to_use.to_list() + ['Negative Control'], 
                        columns=starrfish3_sec1.get_creinfo().index)

for i in cell_types_to_use:
    cres = starrfish3_sec1.get_positive_control_cres(subclass_name_to_subclass[i], use='atac-peak')
    cre_anno.loc[i, cres] = 1
cre_anno.loc['Negative Control', starrfish3_sec1.get_negative_control_cres()] = 1

def plot_heatmap(expression_mat, cell_type_label, cell_types_to_use, cre_anno, scale=None, log=False):
    """
    Plot heatmap of expression matrix across cell types.
    
    Parameters:
    - expression_mat: Expression matrix (cells x CREs)
    - cell_type_label: Cell type labels for each cell
    - cell_types_to_use: Cell types to include in heatmap
    - cre_anno: CRE annotations
    - scale: Scaling factor for expression values
    - log: Whether to log-transform values
    """
    cre_celltype = expression_mat.groupby(cell_type_label).mean()
    
    fig, ax = plt.subplots(nrows=3, figsize=(24, 20), height_ratios=[1, 0.05, 0.05])
    toplot = cre_celltype.loc[cell_types_to_use, cre_whitelist].copy()
    
    if scale is not None:
        toplot = toplot * scale
    if log:
        toplot = np.log10(toplot + 1)
    
    # Sort cell types alphabetically
    toplot = toplot.sort_index()
    
    # Sort CREs by library size (large to small)
    lib_sizes = starrfish3.lib_size['counts'].loc[cre_anno.columns.intersection(cre_whitelist)]
    col_order = lib_sizes.sort_values(ascending=False).index
    toplot_clustered = toplot[col_order]
    
    heatmap = sns.heatmap(toplot_clustered, cmap='coolwarm', ax=ax[0],
                          cbar_kws={'label': 'Expression'})
    
    # Transform colorbar labels back to original scale
    cbar = heatmap.collections[0].colorbar
    cbar_ticks = cbar.get_ticks()
    if log:
        if scale is not None:
            cbar.set_ticklabels([f'{(10 **val - 1) / scale:.3f}' for val in cbar_ticks])
        else:
            cbar.set_ticklabels([f'{10 **val - 1:.3f}' for val in cbar_ticks])
    
    ax[0].set_xticks([])
    
    # Mark library size
    sns.heatmap(np.log10(starrfish3.lib_size['counts'].loc[col_order].values.reshape(1, -1)),
                cmap='coolwarm', ax=ax[1], cbar_kws={'label': 'log10(Library Size)'}, 
                yticklabels=['Library Size'])
    ax[1].set_xticks([])
    
    # Mark negative control
    sns.heatmap(cre_anno.loc[['Negative Control'], col_order], cmap='coolwarm', ax=ax[2])
    ax[2].set_xticks([])
    
    return fig

In [12]:
cell_type_labels = starrfish3.get_tag('obs:subclass_name')

# CRE counts heatmap
fig = plot_heatmap(starrfish3.get_cre_expression(), cell_type_labels, 
                   cell_types_to_use, cre_anno, scale=1000, log=True)
fig.savefig('results/expr3/cre_counts_per_celltype_heatmap.pdf')

# T7 counts heatmap
fig = plot_heatmap(starrfish3.get_t7_expression(), cell_type_labels, 
                   cell_types_to_use, cre_anno, scale=100, log=True)
fig.savefig('results/expr3/t7_counts_per_celltype_heatmap.pdf')

# CRE percentage heatmap
fig = plot_heatmap(starrfish3.get_cre_expression() > 0, cell_type_labels, 
                   cell_types_to_use, cre_anno, scale=1000, log=True)
fig.savefig('results/expr3/cre_percentage_per_celltype_heatmap.pdf')

# T7 percentage heatmap
fig = plot_heatmap(starrfish3.get_t7_expression() > 0, cell_type_labels, 
                   cell_types_to_use, cre_anno, scale=100, log=True)
fig.savefig('results/expr3/t7_percentage_per_celltype_heatmap.pdf')

## 10. Section-to-Section Correlation Analysis

Compute correlations between sections for CRE and T7 expression patterns.

In [13]:
cell_types_to_use = cre_celltype_sec1.index.intersection(cre_celltype_sec2.index)
cres_to_use = cre_whitelist

def plot_corr(activity_df1, activity_df2, cell_types_to_use, cres_to_use, cre_anno, log=False):
    """
    Plot correlation between two activity matrices.
    
    Returns CRE correlations (across cell types) and cell type correlations (across CREs).
    """
    cre_corr, celltype_corr = starrfish3_sec1.corr_starrfish(
        activity_df1=activity_df1.loc[cell_types_to_use, cres_to_use],
        activity_df2=activity_df2.loc[cell_types_to_use, cres_to_use],
        log_activity=log)
    
    cre_corr['lib_size'] = starrfish3_sec1.lib_size['counts'].loc[cre_corr.index]
    celltype_corr['cell_type_size_sec1'] = starrfish3_sec1.get_celltypes().value_counts().loc[
        celltype_corr.index.map(subclass_name_to_subclass)].values
    celltype_corr['cell_type_size_sec2'] = starrfish3_sec2.get_celltypes().value_counts().loc[
        celltype_corr.index.map(subclass_name_to_subclass)].values
    celltype_corr['cell_type_size'] = celltype_corr[['cell_type_size_sec1', 'cell_type_size_sec2']].min(axis=1)
    
    fig, ax = plt.subplots(nrows=2, figsize=(5, 10))
    
    # Plot significant CREs
    significant_cres = cre_corr.index[(cre_corr['pearson_p'] < 0.05) & (cre_corr['pearson'] > 0)].intersection(cres_to_use)
    sns.scatterplot(x=cre_corr.loc[significant_cres, 'lib_size'], 
                    y=cre_corr.loc[significant_cres, 'pearson'], 
                    ax=ax[0], alpha=0.5, label='Significant CREs', color='red')
    
    significant_celltypes = celltype_corr.index[(celltype_corr['pearson_p'] < 0.05) & 
                                                 (celltype_corr['pearson'] > 0)].intersection(cell_types_to_use)
    sns.scatterplot(x=celltype_corr.loc[significant_celltypes, 'cell_type_size'], 
                    y=celltype_corr.loc[significant_celltypes, 'pearson'], 
                    ax=ax[1], alpha=0.5, label='Significant Cell Types', color='red')
    
    # Plot not significant
    not_significant_cres = cre_corr.index[(cre_corr['pearson_p'] >= 0.05) | 
                                          (cre_corr['pearson'] <= 0)].intersection(cres_to_use)
    sns.scatterplot(x=cre_corr.loc[not_significant_cres, 'lib_size'], 
                    y=cre_corr.loc[not_significant_cres, 'pearson'], 
                    ax=ax[0], alpha=0.5, color='blue')
    
    not_significant_celltypes = celltype_corr.index[(celltype_corr['pearson_p'] >= 0.05) | 
                                                     (celltype_corr['pearson'] <= 0)].intersection(cell_types_to_use)
    sns.scatterplot(x=celltype_corr.loc[not_significant_celltypes, 'cell_type_size'], 
                    y=celltype_corr.loc[not_significant_celltypes, 'pearson'], 
                    ax=ax[1], alpha=0.5, color='blue')
    
    ax[1].set_xscale('log')
    ax[0].set_xlabel('log(lib_size)')
    
    return fig

In [14]:
# CRE counts correlation
fig1 = plot_corr(starrfish3_sec1.get_cre_expression().groupby(starrfish3_sec1.get_tag('obs:subclass_name')).sum(), 
                 starrfish3_sec2.get_cre_expression().groupby(starrfish3_sec2.get_tag('obs:subclass_name')).sum(), 
                 cell_types_to_use, cres_to_use, cre_anno, log=False)

# T7 counts correlation
fig2 = plot_corr(starrfish3_sec1.get_t7_expression().groupby(starrfish3_sec1.get_tag('obs:subclass_name')).sum(), 
                 starrfish3_sec2.get_t7_expression().groupby(starrfish3_sec2.get_tag('obs:subclass_name')).sum(), 
                 cell_types_to_use, cres_to_use, cre_anno, log=False)

# CRE proportion correlation
fig3 = plot_corr((starrfish3_sec1.get_cre_expression() > 0).groupby(starrfish3_sec1.get_tag('obs:subclass_name')).mean(), 
                 (starrfish3_sec2.get_cre_expression() > 0).groupby(starrfish3_sec2.get_tag('obs:subclass_name')).mean(), 
                 cell_types_to_use, cres_to_use, cre_anno)

# T7 proportion correlation
fig4 = plot_corr((starrfish3_sec1.get_t7_expression() > 0).groupby(starrfish3_sec1.get_tag('obs:subclass_name')).mean(), 
                 (starrfish3_sec2.get_t7_expression() > 0).groupby(starrfish3_sec2.get_tag('obs:subclass_name')).mean(), 
                 cell_types_to_use, cres_to_use, cre_anno)

fig1.savefig("results/expr3/sec1_sec2.cre.counts.corr.pdf")
fig2.savefig("results/expr3/sec1_sec2.t7.counts.corr.pdf")
fig3.savefig("results/expr3/sec1_sec2.cre.proportion.corr.pdf")
fig4.savefig("results/expr3/sec1_sec2.t7.proportion.corr.pdf")

## 11. Normalized CRE/T7 Ratio Analysis

Analyze CRE/T7 ratios to account for infection efficiency.

In [16]:
cell_types_to_use = cell_type_counts.index[(cell_type_counts['Sec1'] > 1000) & 
                                           (cell_type_counts['Sec2'] > 1000) & 
                                           (cell_type_counts['Exp3'] > 1000)].map(subclass_to_subclass_name)
cres_to_use = starrfish3_sec1.lib_size.index[starrfish3_sec1.lib_size['counts'] >= 7]

# Get cell types with ≥ 0.6 correlation for both CRE and T7
df1 = starrfish3_sec1.get_cre_expression().groupby(starrfish3_sec1.get_tag('obs:subclass_name')).mean().loc[cell_types_to_use]
df2 = starrfish3_sec2.get_cre_expression().groupby(starrfish3_sec2.get_tag('obs:subclass_name')).mean().loc[cell_types_to_use]
cre_corr, celltype_corr = starrfish3_sec1.corr_starrfish(activity_df1=df1[cres_to_use], 
                                                           activity_df2=df2[cres_to_use])

# CRE/T7 mean ratio
plot_corr(starrfish3_sec1.get_cre_expression().groupby(starrfish3_sec1.get_tag('obs:subclass_name')).mean().loc[cell_types_to_use] / 
          starrfish3_sec1.get_t7_expression().groupby(starrfish3_sec1.get_tag('obs:subclass_name')).mean().loc[cell_types_to_use], 
          starrfish3_sec2.get_cre_expression().groupby(starrfish3_sec2.get_tag('obs:subclass_name')).mean().loc[cell_types_to_use] /
          starrfish3_sec2.get_t7_expression().groupby(starrfish3_sec2.get_tag('obs:subclass_name')).mean().loc[cell_types_to_use], 
          cell_types_to_use, cres_to_use, cre_anno)

# CRE/T7 proportion ratio
plot_corr((starrfish3_sec1.get_cre_expression() > 0).groupby(starrfish3_sec1.get_tag('obs:subclass_name')).mean().loc[cell_types_to_use] / 
          (starrfish3_sec1.get_t7_expression() > 0).groupby(starrfish3_sec1.get_tag('obs:subclass_name')).mean().loc[cell_types_to_use], 
          (starrfish3_sec2.get_cre_expression() > 0).groupby(starrfish3_sec2.get_tag('obs:subclass_name')).mean().loc[cell_types_to_use] /
          (starrfish3_sec2.get_t7_expression() > 0).groupby(starrfish3_sec2.get_tag('obs:subclass_name')).mean().loc[cell_types_to_use], 
          cell_types_to_use, cres_to_use, cre_anno)

<Figure size 500x1000 with 2 Axes>

## 12. Infection Rate Filtering Analysis

Analyze how correlation quality depends on the number of infected cells per CRE/cell type combination.

In [17]:
t7_infected_cells_sec1 = (starrfish3_sec1.get_t7_expression() > 0).groupby(starrfish3_sec1.get_tag('obs:subclass_name')).sum()
t7_infected_cells_sec2 = (starrfish3_sec2.get_t7_expression() > 0).groupby(starrfish3_sec2.get_tag('obs:subclass_name')).sum()
cre_infected_cells_sec1 = (starrfish3_sec1.get_cre_expression() > 0).groupby(starrfish3_sec1.get_tag('obs:subclass_name')).sum()
cre_infected_cells_sec2 = (starrfish3_sec2.get_cre_expression() > 0).groupby(starrfish3_sec2.get_tag('obs:subclass_name')).sum()

infected_cells_threshold = 5

def plot_corr_by_infected_cells(activity_df1, activity_df2, infected_cells_sec1, infected_cells_sec2, 
                                celltype_counts_sec1, celltype_counts_sec2, infected_cells_threshold, log=False):
    """
    Compute correlations filtering by minimum number of infected cells.
    
    Returns celltype_corr, cre_corr, and two figures showing correlation vs cell/CRE counts.
    """
    celltype_corr = pd.DataFrame(index=activity_df1.index.intersection(activity_df2.index), 
                                  columns=['pearson', 'p_value', 'n_cres', 'n_cells'])
    
    for cell_type in activity_df1.index.intersection(activity_df2.index):
        cres_to_use = infected_cells_sec1.loc[cell_type].index[
            infected_cells_sec1.loc[cell_type] >= infected_cells_threshold].intersection(
            infected_cells_sec2.loc[cell_type].index[infected_cells_sec2.loc[cell_type] >= infected_cells_threshold]
        ).intersection(cre_whitelist)
        cres_to_use = cres_to_use[~pd.isna(activity_df1.loc[cell_type, cres_to_use])]
        cres_to_use = cres_to_use[~pd.isna(activity_df2.loc[cell_type, cres_to_use])]
        celltype_corr.loc[cell_type, 'n_cres'] = cres_to_use.size
        
        if cres_to_use.size < 2:
            continue
        
        if log:
            corr, p = pearsonr(np.log1p(activity_df1.loc[cell_type, cres_to_use]), 
                               np.log1p(activity_df2.loc[cell_type, cres_to_use]))
        else:
            corr, p = pearsonr(activity_df1.loc[cell_type, cres_to_use], 
                               activity_df2.loc[cell_type, cres_to_use])
        
        celltype_corr.loc[cell_type, 'pearson'] = corr
        celltype_corr.loc[cell_type, 'p_value'] = p
        celltype_corr.loc[cell_type, 'n_cells'] = np.minimum(
            celltype_counts_sec1.loc[cell_type],
            celltype_counts_sec2.loc[cell_type])
    
    cre_corr = pd.DataFrame(index=activity_df1.columns.intersection(activity_df2.columns), 
                            columns=['pearson', 'p_value', 'n_celltypes', 'lib_size'])
    
    for cre in activity_df1.columns.intersection(activity_df2.columns).intersection(cre_whitelist):
        celltypes_to_use = infected_cells_sec1[cre].index[
            infected_cells_sec1[cre] >= infected_cells_threshold].intersection(
            infected_cells_sec2[cre].index[infected_cells_sec2[cre] >= infected_cells_threshold]
        ).intersection(activity_df1.index).intersection(activity_df2.index)
        celltypes_to_use = celltypes_to_use[~pd.isna(activity_df1.loc[celltypes_to_use, cre])]
        celltypes_to_use = celltypes_to_use[~pd.isna(activity_df2.loc[celltypes_to_use, cre])]
        cre_corr.loc[cre, 'n_celltypes'] = celltypes_to_use.size
        
        if celltypes_to_use.size < 2:
            continue
        
        if log:
            corr, p = pearsonr(np.log1p(activity_df1.loc[celltypes_to_use, cre]), 
                               np.log1p(activity_df2.loc[celltypes_to_use, cre]))
        else:
            corr, p = pearsonr(activity_df1.loc[celltypes_to_use, cre], 
                               activity_df2.loc[celltypes_to_use, cre])
        
        cre_corr.loc[cre, 'pearson'] = corr
        cre_corr.loc[cre, 'p_value'] = p
        cre_corr.loc[cre, 'lib_size'] = starrfish3_sec1.lib_size['counts'].loc[cre]
    
    # Plot n_cells with regard to n_cres, colored by pearson
    fig1, ax = plt.subplots(figsize=(6, 6))
    norm = TwoSlopeNorm(vmin=celltype_corr['pearson'].min(), vcenter=0.4, vmax=celltype_corr['pearson'].max())
    sns.scatterplot(x=celltype_corr['n_cells'], y=celltype_corr['n_cres'], 
                    hue=celltype_corr['pearson'], ax=ax, palette='coolwarm', hue_norm=norm)
    ax.set_xscale('log')
    ax.set_xlabel('Number of Cells')
    ax.set_ylabel(f'Number of CREs with infected cells ≥ {infected_cells_threshold}')
    
    # Plot lib size with regard to n_celltypes, colored by pearson
    fig2, ax = plt.subplots(figsize=(6, 6))
    norm = TwoSlopeNorm(vmin=cre_corr['pearson'].min(), vcenter=0.4, vmax=cre_corr['pearson'].max())
    sns.scatterplot(y=cre_corr['n_celltypes'], x=cre_corr['lib_size'], 
                    hue=cre_corr['pearson'], ax=ax, palette='coolwarm', hue_norm=norm)
    ax.set_ylabel(f'Number of Cell Types with infected cells ≥ {infected_cells_threshold}')
    ax.set_xlabel('log(Library Size)')
    
    return celltype_corr, cre_corr, fig1, fig2

In [18]:
celltype_corr, cre_corr, fig1, fig2 = plot_corr_by_infected_cells(
    (starrfish3_sec1.get_cre_expression()).groupby(starrfish3_sec1.get_tag('obs:subclass_name')).mean(),
    (starrfish3_sec2.get_cre_expression()).groupby(starrfish3_sec2.get_tag('obs:subclass_name')).mean(),
    cre_infected_cells_sec1, cre_infected_cells_sec2, 
    starrfish3_sec1.get_tag('obs:subclass_name').value_counts(), 
    starrfish3_sec2.get_tag('obs:subclass_name').value_counts(),
    infected_cells_threshold, log=False)

# Violin plot
cell_type_n_threshold = 20
cre_n_threshold = 50
fig, ax = plt.subplots(figsize=(6, 6))
toplot1 = pd.DataFrame(celltype_corr['pearson'][celltype_corr['n_cres'] >= cre_n_threshold])
toplot2 = pd.DataFrame(cre_corr['pearson'][cre_corr['n_celltypes'] >= cell_type_n_threshold])
toplot1['metric'] = f'within cell type correlation\n{sum(celltype_corr["n_cres"] >= cre_n_threshold)} cell types'
toplot2['metric'] = f'across cell type correlation\n{sum(cre_corr["n_celltypes"] >= cell_type_n_threshold)} CREs'
toplot = pd.concat((toplot1, toplot2))
sns.violinplot(x=toplot['metric'], y=toplot['pearson'], hue=toplot['metric'], ax=ax)

fig1.savefig("results/expr3/sec1_sec2.cre_n>5.cre.within.celltype.corr.pdf")
fig2.savefig("results/expr3/sec1_sec2.cre_n>5.cre.across.celltype.corr.pdf")
fig.savefig("results/expr3/sec1_sec2.cre_n>5.cre.violin.plot.pdf")

## 13. Multiple Threshold Analysis for T7

Test different infection thresholds to see impact on correlation quality.

In [19]:
toplot = pd.DataFrame()

for infected_cells_threshold in [0, 5, 10, 20]:
    celltype_corr, cre_corr, fig1, fig2 = plot_corr_by_infected_cells(
        (starrfish3_sec1.get_t7_expression()).groupby(starrfish3_sec1.get_tag('obs:subclass_name')).mean(),
        (starrfish3_sec2.get_t7_expression()).groupby(starrfish3_sec2.get_tag('obs:subclass_name')).mean(),
        cre_infected_cells_sec1, cre_infected_cells_sec2, 
        starrfish3_sec1.get_tag('obs:subclass_name').value_counts(), 
        starrfish3_sec2.get_tag('obs:subclass_name').value_counts(),
        infected_cells_threshold, log=False)
    
    # Violin plot
    cell_type_n_threshold = 10
    cre_n_threshold = 10
    toplot1 = pd.DataFrame(celltype_corr['pearson'][celltype_corr['n_cres'] >= cre_n_threshold])
    toplot2 = pd.DataFrame(cre_corr['pearson'][cre_corr['n_celltypes'] >= cell_type_n_threshold])
    toplot1['metric'] = f'Cells ≥ {infected_cells_threshold}'
    toplot2['metric'] = f'Cells ≥ {infected_cells_threshold}'
    toplot1['Corr_type'] = 'Cell type wise'
    toplot2['Corr_type'] = 'CRE wise'
    toplot = pd.concat((toplot, toplot1, toplot2), ignore_index=True)

fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(x=toplot['metric'], y=toplot['pearson'], hue=toplot['Corr_type'], ax=ax)
fig.savefig("results/expr3/sec1_sec2.violin.plot.multiple_thresholds.t7.pdf")

## 14. Summary Statistics

Final summary plot showing correlation quality across different metrics.

In [20]:
# Apply infection rate from Expr3
t7_infected_cells_expr3 = (starrfish3.get_t7_expression() > 0).groupby(starrfish3.get_tag('obs:subclass_name')).sum()
t7_infected_cells_expr2 = t7_infected_cells_expr3.copy()
cre_infected_cells_expr3 = (starrfish3.get_cre_expression() > 0).groupby(starrfish3.get_tag('obs:subclass_name')).sum()

infected_cells_threshold = 5
cell_type_n_threshold = 20
cre_n_threshold = 20

fig, ax = plt.subplots(figsize=(6, 6))
toplot1 = pd.DataFrame(celltype_corr['pearson'][celltype_corr['n_cres'] >= cre_n_threshold])
toplot2 = pd.DataFrame(cre_corr['pearson'][cre_corr['n_celltypes'] >= cell_type_n_threshold])
toplot1['metric'] = f'within cell type correlation\n{sum(celltype_corr["n_cres"] >= cre_n_threshold)} cell types'
toplot2['metric'] = f'across cell type correlation\n{sum(cre_corr["n_celltypes"] >= cell_type_n_threshold)} CREs'
toplot = pd.concat((toplot1, toplot2))
sns.violinplot(x=toplot['metric'], y=toplot['pearson'], hue=toplot['metric'], ax=ax)
ax.set_ylabel('Pearson correlation')

fig1.savefig("results/expr3/expr3_expr2.cre_n>5.cre.within.celltype.corr.pdf")
fig2.savefig("results/expr3/expr3_expr2.cre_n>5.cre.across.celltype.corr.pdf")
fig.savefig("results/expr3/expr3_expr2.cre_n>5.cre.violin.plot.pdf")

## Conclusions

This analysis demonstrates:
1. **Strong correlation** between T7 counts and AAV library size
2. **Reproducible cell type distributions** across tissue sections
3. **Consistent CRE and T7 expression** patterns between sections
4. **Improved correlations** when filtering by minimum infected cell counts
5. **Library size effects** on CRE detection reliability

These quality control metrics validate the experimental approach and inform filtering criteria for downstream CRE activity analysis.